# 05 · Building the inlists

This is where the pieces of notebooks 02–04 come together (`posydon/grids/setup/build.py`):

1. `defaults.fill_defaults`: a new-format ini leaves MESA's defaults out; the release's MESA installation fills the `mesa_inlists` layer;
2. `build.layer_targets`: every layer's files per target, with the scenario's files appended to `posydon_inlists`;
3. `build.stack_target`: one `LayerStack` per target;
4. `build.build_inlists`: merge, add the **generated** keys (wiring, formation chain, output options, ZAMS model) and write the files;
5. `explain.explain_text`: the same stacks, as text (`posydon-grid explain`).

In [1]:
from nb_env import env, fresh, write, show, short, grid_csv, tree
ENV = env()

import os
import pandas as pd
from posydon.grids.setup.config import read_config
from posydon.grids.setup.resolve import resolve_scenario
from posydon.grids.setup.defaults import fill_defaults
from posydon.grids.setup.build import build_inlists, layer_targets, legacy_zams, stack_target
from posydon.grids.setup.inlist import read_inlist
from posydon.grids.setup.explain import explain_text

ZAMS = ENV.inlists / "r11701" / "ZAMS_models" / "zams_z1.42m2_y0.2703.data"

def rel(path):
    return short(path, ENV.release, ENV.inlists / "r11701", ENV.mesa, ENV.work)

def make_config(name, grid_type, extra_grid="", extra="", single=False):
    """Write grid.csv and grid.ini into _work/<name>; read the ini."""
    folder = fresh(name)
    grid_csv(folder, single=single)
    ini = write(folder / "grid.ini", f"""
        [grid]
        type = {grid_type}
        points = grid.csv
        run_directory = run
        {extra_grid}
        [scenario]
        release = v2.0
        source = path
        inlist_repo = {ENV.inlists}
        {extra}
        """)
    return read_config(ini), folder

## 1. MESA's defaults are filled in from the release

We use a **HeMS-HMS** grid: star 1 is a helium star built in formation steps, star 2 starts from a ZAMS model. Right after reading the ini, the `mesa_inlists` layer is empty:

In [2]:
config, folder = make_config("05_build", "HeMS-HMS", extra_grid=f"zams_star2 = {ZAMS}")
resolved = resolve_scenario(config)
scenario = resolved.files
print("mesa_inlists before:", config.inlists["mesa_inlists"].targets)
print("extras before:      ", config.extras)

mesa_inlists before: {}
extras before:       {}


In [3]:
fill_defaults(config, resolved, os.environ["MESA_DIR"])
for target, paths in config.inlists["mesa_inlists"].targets.items():
    print(f"{target:38} {', '.join(rel(p) for p in paths)}")
print()
for key, path in config.extras.items():
    print(f"{key:26} {rel(path)}")

star1_formation_every_step_job         mesa/default/defaults/star/star_job.defaults
star1_formation_every_step_controls    mesa/default/defaults/star/controls.defaults
binary_controls                        mesa/default/defaults/binary/binary_controls.defaults
binary_job                             mesa/default/defaults/binary/binary_job.defaults
star1_job                              mesa/default/defaults/star/star_job.defaults
star1_controls                         mesa/default/defaults/star/controls.defaults
star2_job                              mesa/default/defaults/star/star_job.defaults
star2_controls                         mesa/default/defaults/star/controls.defaults

makefile_binary            binary/work/make/makefile
makefile_star              star/work/make/makefile
mesa_binary_extras         binary/work/src/run_binary_extras.f
mesa_star_binary_extras    binary/work/src/run_star_extras.f
mesa_star1_extras          star/work/src/run_star_extras.f
mesa_star2_extras          

Star 1 is formed, so MESA's star defaults also go to `star1_formation_every_step_*`. The extras are MESA's stock sources in `$MESA_DIR`; the scenario's sources replace some of them later (notebook 06). Anything the ini sets itself is kept.

## 2. All layers, per target

`layer_targets` returns every layer's files per target. The scenario's files are appended to `posydon_inlists` (after any `[posydon_inlists]` files of the ini), and `every_step` targets are expanded into each step.

In [4]:
files, flags = layer_targets(config, scenario)
rows = {}
for layer, targets in files.items():
    for target, paths in targets.items():
        rows.setdefault(target, {})[layer] = "\n".join(rel(p) for p in paths)
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows).T.fillna("").style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

,mesa_inlists,posydon_inlists
binary_controls,mesa/default/defaults/binary/binary_controls.defaults,common/binary/inlist_project
binary_job,mesa/default/defaults/binary/binary_job.defaults,common/binary/inlist_project
star1_job,mesa/default/defaults/star/star_job.defaults,common/binary/inlist1 HeMS-HMS/binary/inlist1
star1_controls,mesa/default/defaults/star/controls.defaults,common/binary/inlist1 HeMS-HMS/binary/inlist1
star2_job,mesa/default/defaults/star/star_job.defaults,common/binary/inlist2
star2_controls,mesa/default/defaults/star/controls.defaults,common/binary/inlist2
star1_formation_step0_controls,mesa/default/defaults/star/controls.defaults,common/binary/inlist1 HeMS-HMS/star1_formation/inlist_step1
star1_formation_step1_controls,mesa/default/defaults/star/controls.defaults,common/binary/inlist1 HeMS-HMS/star1_formation/inlist_step2
star1_formation_step0_job,mesa/default/defaults/star/star_job.defaults,common/binary/inlist1 HeMS-HMS/star1_formation/inlist_step1
star1_formation_step1_job,mesa/default/defaults/star/star_job.defaults,common/binary/inlist1 HeMS-HMS/star1_formation/inlist_step2


`flags` holds values that are not in any file. With the old layout this was the single-star flag; release manifests keep it in files (`single_star_flag`), so here it is empty:

In [5]:
flags

{}

## 3. One stack per target

In [6]:
stack = stack_target("star1_formation_step1_controls", files, flags)
merged = stack.merged()
print(len(merged), "keys")
for key in ["max_age", "mesh_delta_coeff"]:
    print(key, [(e["layer"], rel(e["source"]), e["value"]) for e in stack.explain(key)])

1656 keys
max_age [('mesa_inlists', 'mesa/default/defaults/star/controls.defaults', '1d36'), ('posydon_inlists', 'common/binary/inlist1', '1.4e10')]
mesh_delta_coeff [('mesa_inlists', 'mesa/default/defaults/star/controls.defaults', '1.0d0'), ('posydon_inlists', 'common/binary/inlist1', '1.0d0')]


## 4. ZAMS models

`legacy_zams` decides each star's ZAMS model, keeping today's rules plus two fixes:

- a star the scenario builds in formation steps gets **no** ZAMS model, whatever the ini says (fixed S2: before, a HeMS-HMS grid with `zams_filename` never formed star 1);
- a single-star grid whose star 1 is formed has no ZAMS model for star 2 either (fixed S3).

In [7]:
for grid_type, zams_line in [("HMS-HMS", f"zams = {ZAMS}"),
                             ("HeMS-HMS", f"zams = {ZAMS}"),        # zams for both: star 1 still formed (S2)
                             ("CO-HeMS", ""),
                             ("single_HeMS", "")]:
    c, _ = make_config("05_zams", grid_type, extra_grid=zams_line, single=grid_type.startswith("single"))
    z = legacy_zams(c, resolve_scenario(c).files)
    print(f"{grid_type:12} star1: {z['star1'] and rel(z['star1'])!s:45} star2: {z['star2'] and rel(z['star2'])}")

HMS-HMS      star1: ZAMS_models/zams_z1.42m2_y0.2703.data         star2: ZAMS_models/zams_z1.42m2_y0.2703.data
HeMS-HMS     star1: None                                          star2: ZAMS_models/zams_z1.42m2_y0.2703.data
CO-HeMS      star1: None                                          star2: None
single_HeMS  star1: None                                          star2: None


## 5. Write the inlists

`build_inlists` needs the grid's column names: a star inlist that has one of them as a key is wired to the per-point inlist that `posydon-run-grid` writes (`read_extra_controls_inlist1`).

In [8]:
run = folder / "run"
grid_columns = list(pd.read_csv(folder / "grid.csv").columns)
built = build_inlists(config, run, grid_columns, scenario=scenario)
built

BuiltInlists(binary_project='/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/05_build/run/binary/inlist_project', binary_star1='/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/05_build/run/binary/inlist1', binary_star2='/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/05_build/run/binary/inlist2', star1_formation=' /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/05_build/run/star1/inlist_step0 /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/05_build/run/star1/inlist_step1', star2_formation=None)

In [9]:
tree(run)

binary/inlist1
binary/inlist2
binary/inlist_project
star1/inlist_step0
star1/inlist_step1


### What did the setup add itself? (the `generated` layer)

Compare each written file with its merged stack: the difference is what the setup generates.

In [10]:
def generated(path, section, target):
    written = read_inlist(path)[section]
    stacked = stack_target(target, files, flags).merged()
    return {k: (stacked.get(k), v) for k, v in written.items() if stacked.get(k) != v}

for path, section, target in [(run / "binary" / "inlist1", "controls", "star1_controls"),
                              (run / "binary" / "inlist1", "star_job", "star1_job"),
                              (run / "binary" / "inlist2", "controls", "star2_controls"),
                              (run / "binary" / "inlist_project", "binary_job", "binary_job"),
                              (run / "binary" / "inlist_project", "binary_controls", "binary_controls"),
                              (run / "star1" / "inlist_step0", "star_job", "star1_formation_step0_job"),
                              (run / "star1" / "inlist_step1", "star_job", "star1_formation_step1_job")]:
    print(f"{rel(path)}  &{section}")
    for key, (old, new) in generated(path, section, target).items():
        print(f"    {key:40} {old!s:>18} -> {new}")

05_build/run/binary/inlist1  &controls


    read_extra_controls_inlist1                            None -> .true.
    extra_controls_inlist1_name                            None -> 'inlist_grid_star1_binary_controls'
05_build/run/binary/inlist1  &star_job
    save_model_filename                            'model1.mod' -> 'final_star1.mod'
    load_saved_model                                    .false. -> .true.
    saved_model_name                                'undefined' -> 'initial_star1_step1.mod'
05_build/run/binary/inlist2  &controls


    zams_filename                            'zams_z2m2_y28.data' -> '/Users/max/Documents/POSYDON-MESA-INLISTS-v2-base/r11701/ZAMS_models/zams_z1.42m2_y0.2703.data'
    read_extra_controls_inlist1                            None -> .true.
    extra_controls_inlist1_name                            None -> 'inlist_grid_star2_binary_controls'
05_build/run/binary/inlist_project  &binary_job
    inlist_names(1)                                        None -> '/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/05_build/run/binary/inlist1'
    inlist_names(2)                                        None -> '/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/05_build/run/binary/inlist2'
05_build/run/binary/inlist_project  &binary_controls
05_build/run/star1/inlist_step0  &star_job
    save_model_filename                             'step1.mod' -> 'initial_star1_ste

Reading this:

- **wiring**: star 1 and 2 read `inlist_grid_star*_binary_controls` (the per-point values of `initial_z`/`Zbase`); the binary points at its two star inlists;
- **formation chain**: step 0 saves `initial_star1_step0.mod`, step 1 loads it and saves `initial_star1_step1.mod`, and the binary's star 1 loads that;
- **output**: history, final profile and final model per star, from `[output]`;
- **ZAMS**: star 2's `zams_filename`. Star 1 has none: it is formed.

### Formation steps

In [11]:
chain = ["create_pre_main_sequence_model", "load_saved_model", "saved_model_name",
         "save_model_when_terminate", "save_model_filename"]
for path in [run / "star1" / "inlist_step0", run / "star1" / "inlist_step1", run / "binary" / "inlist1"]:
    job = read_inlist(path)["star_job"]
    print(rel(path))
    for key in chain:
        print(f"    {key:32} {job.get(key)}")

05_build/run/star1/inlist_step0
    create_pre_main_sequence_model   .false.
    load_saved_model                 .false.
    saved_model_name                 'undefined'
    save_model_when_terminate        .true.
    save_model_filename              'initial_star1_step0.mod'
05_build/run/star1/inlist_step1
    create_pre_main_sequence_model   .false.
    load_saved_model                 .true.
    saved_model_name                 'initial_star1_step0.mod'
    save_model_when_terminate        .true.
    save_model_filename              'initial_star1_step1.mod'
05_build/run/binary/inlist1
    create_pre_main_sequence_model   .false.
    load_saved_model                 .true.
    saved_model_name                 'initial_star1_step1.mod'
    save_model_when_terminate        .true.
    save_model_filename              'final_star1.mod'


## 6. Explain

`explain_text` prints the same stacks: every key whose value does not come from MESA's defaults alone. Generated keys are not in it; they come on top when the files are written.

In [12]:
text = explain_text(config, scenario, roots=[ENV.inlists])
print("\n".join(text.splitlines()[:30]))
print("...")

binary_controls
  history_interval = 1  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist_project
      over 5  [mesa_inlists] r11701/mesa/default/defaults/binary/binary_controls.defaults
  append_to_star_history = .false.  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist_project
      over .true.  [mesa_inlists] r11701/mesa/default/defaults/binary/binary_controls.defaults
  photo_interval = 0  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist_project
      over 50  [mesa_inlists] r11701/mesa/default/defaults/binary/binary_controls.defaults
  terminal_interval = 10  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist_project
      over 1  [mesa_inlists] r11701/mesa/default/defaults/binary/binary_controls.defaults
  write_header_frequency = 100  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist_project
      over 10  [mesa_inlists] r11701/mesa/default/defaults/binary/binary_controls.defaults
  fm = 5d-3  [posydon_inlists] r11701/releases/

## 7. Add your own inlist and see what changes

In [13]:
write(folder / "my_inlist2", """
    &controls
        mesh_delta_coeff = 0.8d0
        initial_z = 0.0142d0
    /
    """)
config2, _ = make_config("05_build_user", "HeMS-HMS", extra_grid=f"zams_star2 = {ZAMS}",
                         extra=f"[user_inlists]\nstar2 = {folder / 'my_inlist2'}")
resolved2 = resolve_scenario(config2)
fill_defaults(config2, resolved2, os.environ["MESA_DIR"])
run2 = ENV.work / "05_build_user" / "run"
build_inlists(config2, run2, grid_columns, scenario=resolved2.files)

a = read_inlist(run / "binary" / "inlist2")["controls"]
b = read_inlist(run2 / "binary" / "inlist2")["controls"]
{k: (a.get(k), v) for k, v in b.items() if a.get(k) != v}

{'initial_z': ('0.02d0', '0.0142d0'), 'mesh_delta_coeff': ('1.0d0', '0.8d0')}

## Single-star grids

A single-star grid runs star 1 alone, as its "formation" steps; the binary inlists are written but empty. The grid type's `single_star_flag` (`x_logical_ctrl(1) = .true.`) tells `run_star_extras.f` to evolve it on its own.

In [14]:
c, f = make_config("05_single", "single_HMS", extra_grid=f"zams = {ZAMS}", single=True)
r = resolve_scenario(c)
fill_defaults(c, r, os.environ["MESA_DIR"])
b = build_inlists(c, f / "run", ["initial_z", "Zbase", "initial_mass"], scenario=r.files)
print(b)
show(f / "run" / "star1" / "inlist_step0", grep="x_logical_ctrl(1)")
show(f / "run" / "star1" / "inlist_step0", grep="zams_filename")

BuiltInlists(binary_project='/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/05_single/run/binary/inlist_project', binary_star1='/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/05_single/run/binary/inlist1', binary_star2='/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/05_single/run/binary/inlist2', star1_formation=' /Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/05_single/run/star1/inlist_step0', star2_formation=None)
	x_logical_ctrl(1) = .true.
	zams_filename = '/Users/max/Documents/POSYDON-MESA-INLISTS-v2-base/r11701/ZAMS_models/zams_z1.42m2_y0.2703.data'


## Try it

- Build `CO-HMS_RLO` and check that `binary/inlist2` has no `zams_filename` (star 2 is a point mass).
- Set `[output] star1 = none` and see `do_history_file` and `save_model_when_terminate` change.
- Add `[posydon_inlists] star1 = <file>`: is it applied before or after the release's files? (`layer_targets` answers it.)